<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/FrozenLake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Aprendizaje por Refuerzo: Q-Learning con Frozen Lake**
## **Cómo aprende un agente experimentando**

## Índice

### 1. Introducción: Aprender por Ensayo y Error

* ¿Qué es el Reinforcement Learning (RL)?
* Diferencias con el Aprendizaje Supervisado (aprender de un jefe vs. aprender de la experiencia).
* El ciclo Agente ↔ Entorno ↔ Recompensa.

### 2. El Escenario: Frozen Lake y Gymnasium

* Presentación del problema: Hielo, agujeros y una meta.
* El concepto de **Entorno Estocástico**: ¿Por qué a veces el agente resbala?
* **💻 Código:** Instalación de `gymnasium` y creación del entorno.

### 3. Conceptos Clave: El Vocabulario del RL

* Estado, Acción y Recompensa.
* La Política (π): El "manual de instrucciones" que el agente debe escribir.
* El objetivo: Maximizar la recompensa acumulada.

### 4. La Q-Table: La Memoria del Agente

* ¿Qué es realmente una Q-Table? Una "hoja de trucos" de 16×4.
* **💻 Código:** Inicialización de la tabla con `numpy.zeros`.

### 5. La Ecuación de Bellman: ¿Cómo aprende el cerebro?

* Explicación intuitiva: "Actualizo lo que creía saber con lo que acabo de descubrir".
* La fórmula en LaTeX.
* Explicación de los "mandos de control": α (alfa) ritmo de aprendizaje y γ (gamma) visión a futuro.

### 6. Explorar o Explotar: El Dilema -greedy

* El dilema del agente: ¿Hago lo que ya sé que funciona o pruebo algo nuevo?
* Implementación del decaimiento de epsilon (ε-decay).

### 7. ¡A Entrenar! Enseñando al Agente a Sobrevivir

* **💻 Código:** El bucle de entrenamiento completo.
* Explicación de los pasos: `reset`, `step`, `update`, `render`.

### 8. Análisis de Resultados: ¿Ha aprendido nuestro agente?

* **💻 Código:** Gráfica de la curva de aprendizaje (recompensas por episodio).
* **💻 Código:** Visualización pro mediante un **Heatmap** de la Q-Table (¿Qué zonas considera peligrosas?).
* **💻 Código:** Demo: Ver al agente jugar.

### 9. Conclusión y Siguientes Pasos

* Limitaciones de las tablas (¿Qué pasa si el mundo es gigante?).
* Introducción al Deep Q-Learning (DQN) para triunfar en los juegos de Atari.

---


# **1. Introducción: Aprender por Ensayo y Error**

¡Bienvenido al fascinante mundo del **Reinforcement Learning (RL)** o Aprendizaje por Refuerzo! Si alguna vez has entrenado a un perro para que se siente a cambio de una golosina, o si has aprendido a montar en bicicleta cayéndote una cuantas veces antes de mantener el equilibrio, ya entiendes los principios básicos de esta rama de la Inteligencia Artificial.

### ¿Qué es el Reinforcement Learning?

A diferencia de otras áreas de la IA, el RL no se trata de predecir una etiqueta (como decir si una foto es un gato o un perro) ni de encontrar patrones ocultos en los datos. El RL trata sobre la **toma de decisiones secuenciales**.

El **Reinforcement Learning** es una rama del Machine Learning donde un **Agente** aprende a tomar decisiones interactuando con un **Entorno**. A diferencia de otros métodos de aprendizaje automático, el Agente:

- ❌ **No tiene un profesor que le diga exactamente qué hacer en cada situación**
- ✅ **Aprende de las consecuencias de sus propias acciones**
- ✅ **Recibe señales de "recompensa" o "castigo" que le indican si lo está haciendo bien o mal**

Es como aprender a jugar al ajedrez: nadie te dice "en esta posición exacta, mueve el caballo aquí". En cambio, juegas muchas partidas, pierdes, ganas, y poco a poco descubres qué estrategias funcionan mejor.

---

### Aprendizaje Supervisado vs. RL: El Jefe vs. La Experiencia

Para entenderlo mejor, comparemos el RL con el aprendizaje más tradicional:

* **Aprendizaje Supervisado (El "Jefe"):** Imagina que tienes un jefe que te da una lista de tareas y te dice exactamente cómo debe quedar cada una. Si te equivocas, te corrige al instante. Tienes ejemplos claros de "entrada" y "salida".
* **Reinforcement Learning (La "Experiencia"):** Aquí no hay jefe. Estás solo en una habitación con un objetivo. Pruebas cosas: si algo funciona, recibes una moneda; si algo sale mal, no recibes nada o recibes un pequeño "toque". Nadie te dice *qué* hacer, solo te dicen *qué tan bien* lo hiciste después de que lo intentaste.

> **En resumen:** En el aprendizaje supervisado aprendes de un "maestro". En el RL, aprendes de tu propia **experiencia**.


| **Aprendizaje Supervisado** | **Aprendizaje por Refuerzo** |
|------------------------------|------------------------------|
| **"Aprender de un profesor"** | **"Aprender de la experiencia"** |
| Tienes datos etiquetados: "esta imagen es un gato" | No hay etiquetas, solo señales de recompensa |
| El objetivo es **imitar** ejemplos correctos | El objetivo es **descubrir** la mejor estrategia |
| Aprendes de datos estáticos | Aprendes interactuando con un entorno dinámico |
| Ejemplo: Clasificar imágenes de perros y gatos | Ejemplo: Enseñar a un robot a caminar |

---


### El ciclo fundamental del RL

Un sistema de Reinforcement Learning logra que el modelo aprenda iterando este ciclo.

```
    ┌─────────────────────────────────────────┐
    │                                         │
    │   👤 AGENTE                             │
    │   (Quien toma las decisiones)           │
    │                                         │
    └──────────┬──────────────────────────────┘
               │                    ▲
               │ Acción             │ Estado +
               │                    │ Recompensa
               ▼                    │
    ┌───────────────────────────────┴─────────┐
    │                                         │
    │   🌍 ENTORNO                            │
    │   (El mundo donde actúa el agente)      │
    │                                         │
    └─────────────────────────────────────────┘
```


#### Los cinco componentes fundamentales:

1. **🤖 El AGENTE** (Agent)
   - Es quien aprende y toma decisiones
   - En nuestro caso: el algoritmo de Q-Learning
   - Piensa en él como "el jugador"

2. **🌍 El ENTORNO** (Environment)
   - Es el mundo donde actúa el agente
   - Define las reglas del juego
   - En nuestro caso: el lago congelado de Frozen Lake
   - Puede ser determinista (siempre pasa lo mismo) o estocástico (tiene aleatoriedad)

3. **📍 El ESTADO** (State - *s*)
   - Es la situación actual en la que se encuentra el agente
   - Representa "dónde estoy" o "qué está pasando ahora"
   - En Frozen Lake: La casilla específica donde está el agente
   - Ejemplo: _"Estoy en la casilla (2,3)"_

4. **🎯 La ACCIÓN** (Action - *a*)
   - Es la decisión que toma el agente
   - Representa "qué voy a hacer"
   - En Frozen Lake: Moverse en una de cuatro direcciones (←, ↓, →, ↑)
   - Ejemplo: _"Voy a moverme hacia arriba"_

5. **🎁 La RECOMPENSA** (Reward - *r*)
   - Es la señal de feedback que recibe el agente después de cada acción
   - Le dice "esto estuvo bien" (+) o "esto estuvo mal" (-)
   - En Frozen Lake: +1 por llegar a la meta, 0 en cualquier otro caso
   - Es el "profesor silencioso" que guía el aprendizaje

<img src="https://github.com/financieras/math_for_ai/blob/main/img/reinforcement_learning.jpeg?raw=1" alt="reinforcement learning" width="480"/>

---

#### ¿Cómo funciona el ciclo completo?

Ahora que conocemos los componentes, veamos cómo interactúan en cada paso:

1. **📍 El Agente observa el ESTADO actual** del entorno
   - _"Estoy en la casilla (0,0) del lago"_

2. **🎯 El Agente elige una ACCIÓN** basándose en su conocimiento actual y se ejecuta la acción.
   - _"Voy a moverme hacia la derecha"_

3. **🌍 El Entorno responde**:
   - Cambia el Estado
   - Hay un nuevo **ESTADO**: _"Ahora estás en la casilla (0,1)"_
   - Le da una **RECOMPENSA**  (puede ser positiva, negativa o cero):
   - _"0 puntos (aún no llegas a la meta)"_

4. **🤖 El Agente aprende** de esta experiencia
   - Actualiza su conocimiento: _"Ah, moverme a la derecha desde (0,0) me llevó a (0,1) sin obtener recompensa"_

5. **🔄 Se repite el ciclo** desde el paso 1 con el nuevo estado
   - El agente continúa hasta alcanzar un **estado terminal**: llegar a la meta 🎯 o caer en un agujero ❌

---

Este ciclo completo (desde el inicio hasta un estado terminal) se llama un **episodio**. El agente jugará **miles de episodios**, y con cada uno se volverá más inteligente al descubrir qué acciones llevan a mejores recompensas. Así el agente descubre una buena estrategia (lo que llamamos política).

**Piénsalo así:**
- Un episodio = Una partida completa del juego
- Miles de episodios = El entrenamiento completo del agente

---

### ¿Listo para empezar?

Un agente deberá cruzar un lago congelado evitando agujeros… ¡sin que nadie le diga por dónde ir!
- Solo recibirá una recompensa +1 cuando llegue a la meta… y -1 si cae al agua.  
- Todo lo demás lo tendrá que aprender **por ensayo y error**.


🧊 ¡Vamos a patinar sobre hielo! 🧊

---

# **2. El Escenario: Frozen Lake y Gymnasium**

Ahora que comprendes la filosofía del Reinforcement Learning, es momento de conocer el **campo de entrenamiento** donde nuestro agente aprenderá a sobrevivir: el entorno de un lago congelado lleno de peligros.

---

## 🧊 El Problema: Un Lago Peligroso

Imagina este escenario:

> *Eres un explorador que debe cruzar un lago congelado para recuperar un regalo 🎁 que está en la esquina opuesta. Empiezas en una esquina (S = Start) y necesitas llegar a donde está el regalo (G = Goal). Pero hay un problema: el hielo está agrietado y hay varios agujeros (H = Hole) que debes evitar a toda costa. Además, el hielo es resbaladizo, así que cuando intentas moverte en una dirección... ¡a veces te deslizas hacia otra!*

El mapa del lago se ve así (versión 4×4):

```
S F F F
F H F H
F F F H
H F F G
```

**Leyenda:**
- **S** (Start): Punto de inicio - Casilla segura 🚶
- **F** (Frozen): Hielo congelado - Casilla segura pero puede resbalar 🧊
- **H** (Hole): Agujero - ¡Si caes aquí, pierdes! ❌
- **G** (Goal): Meta - ¡Si llegas aquí, ganas! 🎯

**Las 4 Acciones Posibles:**

Nuestro agente solo puede moverse en **cuatro direcciones**. En la librería que utilizaremos (`gymnasium`) las acciones están mapeadas a estos números:

0. Left
1. Down
2. Right
3. Up

<img src="https://github.com/financieras/math_for_ai/blob/main/img/frozen_lake.gif?raw=1" alt="frozen lake" width="320"/>

---

## 🎯 Sistema de Recompensas

El sistema de recompensas en Frozen Lake es muy simple:

| Situación | Recompensa | ¿Termina el episodio? |
|-----------|------------|------------------------|
| Alcanzar la meta (**G**) | **+1** | ✅ Sí |
| Caer en un agujero (**H**) | 0 | ✅ Sí |
| Pisar hielo seguro (**F** o **S**) | 0 | ❌ No |

> 🎯 **Objetivo del agente:** Maximizar la recompensa acumulada. Como solo hay +1 al final, ¡el agente debe aprender a llegar a G en el menor número de pasos posible!

**Nota importante:**  
Aunque caer en un agujero da recompensa 0, el episodio termina inmediatamente, por lo que el agente aprenderá a evitarlos (ya que no puede seguir acumulando recompensas futuras).

---

## 🎲 Entorno Estocástico: El Factor Sorpresa

Aquí viene lo interesante (y frustrante): **Frozen Lake es un entorno estocástico**.

### ¿Qué significa "estocástico"?

Significa que hay **aleatoriedad** en el entorno. Por defecto, Frozen Lake es **estocástico** (`is_slippery=True`). Cuando el agente intenta moverse en una dirección:
- **1/3 de probabilidad** → Va en la dirección que eligió
- **1/3 de probabilidad** → Va perpendicular a un lado
- **1/3 de probabilidad** → Va perpendicular al otro lado

* **En un entorno determinista:** Si ordenas "moverte a la derecha", te mueves exactamente a la derecha.
* **En Frozen Lake (estocástico):** Si ordenas "moverte a la derecha", el hielo resbaladizo puede hacer que:
    * Te muevas a la derecha (como querías) con una probabilidad de **1/3**.
    * Te muevas en una dirección adyacente (arriba o abajo) con una probabilidad de **1/3** cada una.

**¿Por qué es importante esto?** Porque fuerza al agente a aprender políticas **robustas**, no solo memorizar una secuencia de pasos. Debe aprender a recuperarse de los resbalones y evitar acercarse demasiado a los agujeros, porque un pequeño desliz podría ser fatal.

#### Nota técnica  
- Estas probabilidades están definidas internamente en el código fuente de Gymnasium y no pueden modificarse mediante parámetros.
- Si tu posición está junto a un borde y intentas moverte hacia él, te quedarás en el mismo sitio, lo que puede alterar las probabilidades efectivas dependiendo de tu posición.
---

## 🐍 ¡Manos al código! Instalación y Configuración

Vamos a usar **[Gymnasium](https://gymnasium.farama.org/index.html)** (anteriormente conocido como OpenAI Gym), que es la biblioteca estándar para entornos de RL.

### Paso 1: Instalación

```python
# Instalamos Gymnasium
!pip install gymnasium -q

# Importamos las librerías necesarias
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

print("✅ Librerías importadas correctamente")
```

---

### Paso 2: Creación del Entorno

```python
# Creamos el entorno Frozen Lake
# is_slippery=True activa el modo estocástico (con resbalones)
env = gym.make(
    'FrozenLake-v1',
    map_name="4x4",      # Especifica el tamaño del mapa. Existe otro 8x8
    is_slippery=True,    # Activa el comportamiento estocástico
    render_mode='ansi'   # Modo de visualización en texto
)

print("🧊 Entorno Frozen Lake creado")
print(f"📊 Tamaño del espacio de estados: {env.observation_space.n}")
print(f"🎮 Tamaño del espacio de acciones: {env.action_space.n}")
```

**Salida esperada:**
```
🧊 Entorno Frozen Lake creado
📊 Tamaño del espacio de estados: 16
🎮 Tamaño del espacio de acciones: 4
```

**Explicación:**
- **16 estados**: Una casilla para cada posición del tablero 4×4
- **4 acciones**: Izquierda (0), Abajo (1), Derecha (2), Arriba (3)

---

### Paso 3: Explorando el Entorno

Vamos a ver cómo se ve el lago y entender la numeración de los estados:

```python
# Con 'reset' reiniciamos el entorno para empezar un nuevo episodio
state, info = env.reset()

print("🎬 Estado inicial del lago:")
print(env.render())
print(f"\n📍 Posición inicial (estado): {state}")
```

**Salida esperada:**
```
🎬 Estado inicial del lago:

SFFF
FHFH
FFFH
HFFG

📍 Posición inicial (estado): 0
```

### Numeración de Estados (Casillas del tablero)

```
 0   1   2   3      S   F   F   F
 4   5   6   7  →   F   H   F   H
 8   9  10  11      F   F   F   H
12  13  14  15      H   F   F   G
```

El estado 0 es la esquina superior izquierda (S), y el estado 15 es la esquina inferior derecha (G).

---

### Paso 4: Probando Acciones Aleatorias

Veamos qué pasa cuando el agente toma acciones al azar:

```python
# Diccionario para traducir las acciones
action_names = {0: "← Izquierda", 1: "↓ Abajo", 2: "→ Derecha", 3: "↑ Arriba"}

def get_expected_state(current_state, action):
    """Calcula el estado esperado sin resbalones (tablero 4x4)."""
    row, col = current_state // 4, current_state % 4
    
    if action == 0:    col = max(col - 1, 0)
    elif action == 1:  row = min(row + 1, 3)
    elif action == 2:  col = min(col + 1, 3)
    elif action == 3:  row = max(row - 1, 0)
    
    return row * 4 + col

current_state, _ = env.reset()
print("🎲 Tomando 100 acciones aleatorias...\n")

# Encabezado de la tabla
print(f"{'Paso':<6} {'Estado':<8} {'Acción':<12} {'Estado':<8} {'Recompensa':<11} {'¿Resbaló?':<10}")
print(f"{'':6} {'Actual':<8} {'Elegida':<12} {'Nuevo':<8} {'':<11} {'':<10}")
print("=" * 70)

for i in range(100):
    action = env.action_space.sample()
    expected_state = get_expected_state(current_state, action)
    new_state, reward, terminated, truncated, info = env.step(action)
    
    slip = "🛷 Sí" if new_state != expected_state else "✅ No"
    
    print(f"{i+1:<6} {current_state:<8} {action_names[action]:<12} {new_state:<8} {reward:<11} {slip:<10}")
    
    if terminated or truncated:
        print("=" * 70)
        print(f"🏁 Episodio terminado: {'🎉 ¡META ALCANZADA!' if reward == 1 else '💀 Agujero'}")
        break
    
    current_state = new_state

env.close()     # Cerramos el entorno
```

**Salida aleatoria posible:**

```
🎲 Tomando 100 acciones aleatorias...

Paso   Estado   Acción       Estado   Recompensa  ¿Resbaló?
       Actual   Elegida      Nuevo                          
============================================================
1      0        ↓ Abajo      4        0           ✅ No      
2      4        ← Izquierda  8        0           🛷 Sí      
3      8        ← Izquierda  8        0           ✅ No      
4      8        ↑ Arriba     8        0           🛷 Sí      
5      8        → Derecha    9        0           ✅ No      
6      9        ← Izquierda  5        0           🛷 Sí      
============================================================
🏁 Episodio terminado: 💀 Agujero
```

**Una posible salida con éxito:**

```
🎲 Tomando 100 acciones aleatorias...

Paso   Estado   Acción       Estado   Recompensa  ¿Resbaló?
       Actual   Elegida      Nuevo                          
======================================================================
1      0        ← Izquierda  0        0           ✅ No      
2      0        → Derecha    1        0           ✅ No      
3      1        ↑ Arriba     2        0           🛷 Sí      
4      2        ↓ Abajo      6        0           ✅ No      
5      6        ← Izquierda  2        0           🛷 Sí      
6      2        ← Izquierda  2        0           🛷 Sí      
7      2        → Derecha    2        0           🛷 Sí      
8      2        ← Izquierda  6        0           🛷 Sí      
9      6        ↓ Abajo      10       0           ✅ No      
10     10       ↓ Abajo      14       0           ✅ No      
11     14       → Derecha    15       1           ✅ No      
======================================================================
🏁 Episodio terminado: 🎉 ¡META ALCANZADA!
```
> **Resultado típico:** El agente aleatorio casi siempre termina en un agujero o da vueltas sin sentido hasta que se agotan los pasos permitidos. De miles de episodios aleatorios, apenas unos pocos llegan a la meta por pura suerte.

> **💡 Observa:** El método `env.step(action)` devuelve 5 valores:
> - `new_state`: El nuevo estado: la casilla donde terminó el agente
> - `reward`: La recompensa obtenida: +1 solo al llegar a la meta
> - `terminated`: True si el episodio acabó (llegó a G o cayó en H)
> - `truncated`: True si se excedió el límite de pasos
> - `info`: Información adicional del entorno

---

## 🤔 Reflexión: ¿Por qué es difícil este problema?

Detengámonos un momento a pensar en los desafíos que enfrenta nuestro agente:

1. **🎲 Aleatoriedad**: El hielo resbaladizo hace que las acciones sean impredecibles
2. **🕳️ Agujeros mortales**: Un movimiento en falso y se acabó el episodio
3. **🎯 Recompensa diferida**: Solo obtienes +1 al llegar a la meta, no hay "pistas" en el camino
4. **🗺️ Caminos múltiples**: Hay varias rutas posibles para llegar a la meta

**Sin embargo**, nuestro agente de Q-Learning aprenderá a:
- Identificar qué acciones son más seguras en cada casilla
- Encontrar el camino óptimo considerando el riesgo de resbalones
- Maximizar la probabilidad de llegar a la meta

---

## ✅ Recapitulación

Hasta ahora hemos aprendido:

✔️ **Frozen Lake** es un lago congelado con agujeros donde debemos llegar a la meta  
✔️ Es un **entorno estocástico**: las acciones no siempre tienen el resultado esperado  
✔️ Tenemos **16 estados** (casillas) y **4 acciones** (direcciones)  
✔️ La **recompensa** solo se obtiene al llegar a la meta (+1)  
✔️ Gymnasium facilita la creación y manipulación de este entorno

---

### 🚀 Siguiente Paso

Ahora que conocemos nuestro campo de batalla, es momento de profundizar en los **conceptos clave del RL**: Estado, Acción, Recompensa y, lo más importante, la **Política** que nuestro agente debe aprender.

En la siguiente sección descubriremos qué significa realmente "aprender una política" y cómo el agente decide qué hacer en cada situación.

❄️ ¿Listo para seguir patinando? ❄️

---

# **3. Conceptos Clave: El Vocabulario del RL**

Ya vimos a nuestro agente tropezando sobre el hielo, tomando decisiones al azar y cayendo en agujeros una y otra vez. Antes de enseñarle a ser más inteligente, necesitamos entender el lenguaje que usaremos. No te preocupes, no vamos a llenarte de fórmulas matemáticas, sino que explicaremos cada concepto de forma que tenga sentido intuitivo.

Piensa en esta sección como el "diccionario básico" que necesitas para entender cómo funciona realmente el aprendizaje por refuerzo.

---

## 📍 Estado (State)

El **Estado** es simplemente: **¿Dónde está el agente ahora?**

**En Frozen Lake:**
- El estado es un número del 0 al 15 que indica en qué casilla está el agente
- Estado 0 = esquina superior izquierda (S - Start)
- Estado 15 = esquina inferior derecha (G - Goal)
- Es muy simple: solo necesitas saber en qué casilla estás

```
 0   1   2   3      S   F   F   F
 4   5   6   7  →   F   H   F   H
 8   9  10  11      F   F   F   H
12  13  14  15      H   F   F   G
```

**En otros problemas de RL:**
- En un videojuego de carreras: tu velocidad, posición en la pista, distancia a otros coches
- En un robot que camina: ángulos de sus articulaciones, velocidad, si está en equilibrio
- En un jugador de ajedrez: la posición actual de todas las piezas en el tablero

> 💡 **Idea clave:** El estado es toda la información que el agente necesita para tomar una buena decisión. En el ajedrez no necesitas saber cómo llegaste a esa posición del tablero, solo necesitas ver dónde están las piezas ahora.

---

## 🎯 Acción (Action)

La **Acción** es: **¿Qué puede hacer el agente?**

**En Frozen Lake:**
- Tenemos exactamente 4 acciones posibles:
  - 0 = ← Izquierda
  - 1 = ↓ Abajo
  - 2 = → Derecha
  - 3 = ↑ Arriba
- Son las mismas 4 acciones sin importar en qué casilla estés

**Las acciones pueden ser:**
- **Discretas** (opciones separadas): como los botones de un mando de videojuegos
  - Frozen Lake: 4 direcciones
  - Pac-Man: arriba, abajo, izquierda, derecha, quedarse quieto
- **Continuas** (valores en un rango): como girar un volante
  - Conducir un coche: girar el volante cualquier ángulo entre -45° y +45°
  - Robot que lanza objetos: aplicar fuerza entre 0 y 100 Newtons

> 💡 **Piénsalo así:** Las acciones son los "botones" que el agente puede pulsar para interactuar con el mundo.

---

## 🎁 Recompensa (Reward)

La **Recompensa** es: **¿Qué tan bien lo hice?**

Es la señal de feedback que recibe el agente después de tomar una acción. Es lo único que le indica si va por buen camino o no.

**En Frozen Lake:**
```
Llegar a la meta (G):  r = +1  ✅
Caer en agujero (H):   r = 0   💀 (pero el juego termina)
Cualquier otro paso:   r = 0   ❄️
```

**¿Por qué es difícil aprender aquí?**

Imagina que estás aprendiendo a cocinar y tu profesor te dice:
- Si haces el plato perfecto: "¡Excelente! 10 puntos"
- Si lo quemas todo: Silencio (0 puntos)
- Si vas por buen camino pero aún no terminas: Silencio (0 puntos)

¡Es muy difícil saber si vas bien! Esto se llama **recompensa escasa** (sparse reward).

**Comparación:**

```
🔴 RECOMPENSA ESCASA (Frozen Lake actual):
   Llegar a la meta: +1
   Todo lo demás: 0
   
   Problema: El agente no sabe si va bien hasta el final

🟢 RECOMPENSA DENSA (más fácil de aprender):
   Llegar a la meta: +100
   Cada paso hacia la meta: +1
   Cada paso alejándose: -1
   Caer en agujero: -50
   
   Ventaja: Feedback constante, aprendizaje más rápido
```

> ⚠️ **Cuidado:** La recompensa es solo una "pista momentánea". El verdadero objetivo del agente no es maximizar la recompensa inmediata, sino la **suma total de recompensas futuras**. Es como en la vida: a veces tienes que hacer sacrificios ahora (recompensa 0) para obtener algo mejor después (recompensa +1).

---

## 📋 La Política (Policy): El "Manual de Supervivencia"

Aquí llegamos al concepto **más importante** de todo el Reinforcement Learning.

### ¿Qué es una Política?

La **Política** es el "manual de instrucciones" del agente. Es la regla que le dice **qué acción tomar en cada estado**.

Imagina que escribes un manual para un amigo que va a cruzar el lago congelado:

```
📖 MANUAL DE SUPERVIVENCIA EN EL LAGO ❄️

Si estás en la casilla 0 (inicio)      → Ve a la DERECHA
Si estás en la casilla 1               → Ve a la DERECHA
Si estás en la casilla 2               → Ve ABAJO
Si estás en la casilla 6               → Ve ABAJO
Si estás en la casilla 10              → Ve ABAJO
Si estás en la casilla 14              → Ve a la DERECHA
¡Llegas a la meta! 🎯
```

Eso es una **política**. En cada situación (estado), te dice qué hacer (acción).

Este camino (0→1→2→6→10→14→15) es uno de los posibles, pero hay otros, por ejemplo 0→4→8→9→13→14→15.

### Tipos de Políticas

**1. Política Determinista** (siempre hace lo mismo)

```
En el estado 0 → SIEMPRE ve a la derecha
En el estado 1 → SIEMPRE ve abajo
```

Es como un robot que siempre sigue exactamente el mismo plan.

**2. Política Estocástica** (elige con probabilidades)

```
En el estado 0:
  - Ir a la derecha: 80%
  - Ir abajo: 15%
  - Ir arriba: 4%
  - Ir a la izquierda: 1%
```

Es más flexible, como un jugador de póker que varía su estrategia.

### La Política Óptima: El Santo Grial

El **objetivo de todo el Reinforcement Learning** es encontrar la **política óptima**: aquella que maximiza la recompensa total que el agente puede obtener.

**En Frozen Lake, la política óptima sería aquella que:**
- Nos lleva a la meta con la mayor probabilidad posible
- Evita los agujeros mortales
- Tiene en cuenta que el hielo es resbaladizo (las acciones no siempre funcionan como esperamos)

> 🎯 **La gran pregunta:** ¿Cómo encuentra el agente esta política óptima sin que nadie se la diga? ¡Esa es precisamente la magia del Q-Learning que veremos pronto!

---

## 💰 La Recompensa Acumulada: Pensar en el Futuro

Aquí está uno de los conceptos más sutiles pero importantes del RL.

### El Agente Impulsivo vs. El Agente Inteligente

Imagina dos tipos de agentes:

**🔴 Agente Impulsivo (miope)**
- Solo le importa la recompensa **inmediata**
- Decisión: "Si todas las acciones me dan 0 ahora... ¡me da igual cuál elegir!"
- Resultado: Se mueve al azar, nunca aprende

**🟢 Agente Inteligente (con visión)**
- Piensa en la **suma de todas las recompensas futuras**
- Decisión: "Este movimiento me da 0 ahora, pero me acerca a la meta donde obtendré +1"
- Resultado: Encuentra estrategias que funcionan

### La Recompensa Acumulada (Return)

El **Return** o retorno es simplemente la suma de todas las recompensas desde ahora hasta el final:

**Ejemplo real:**
```
Episodio exitoso:
Paso 1: Estado 0 → Derecha → Estado 1, Recompensa: 0
Paso 2: Estado 1 → Derecha → Estado 2, Recompensa: 0
Paso 3: Estado 2 → Abajo  → Estado 6, Recompensa: 0
Paso 4: Estado 6 → Abajo  → Estado 10, Recompensa: 0
Paso 5: Estado 10 → Abajo → Estado 14, Recompensa: 0
Paso 6: Estado 14 → Derecha → Estado 15 (META), Recompensa: +1

Retorno total = 0 + 0 + 0 + 0 + 0 + 1 = 1 punto 🎉
```

### El Factor de Descuento (γ - gamma): ¿Cuánto Valoras el Futuro?

Aquí hay un problema: si un episodio es muy largo, las recompensas lejanas pierden importancia. Es como el dinero: prefieres 100€ hoy que 100€ dentro de 10 años.

Para esto usamos **gamma** (γ), un número entre 0 y 1 que controla cuánto valora el agente el futuro:

```
γ = 0.0  →  Agente super impulsivo: "Solo me importa YA"
γ = 0.9  →  Agente equilibrado: "Valoro el futuro, pero no tanto"
γ = 0.99 →  Agente muy paciente: "Sacrifico ahora por ganar después"
```

**Ejemplo intuitivo:**

Si γ = 0.9, una recompensa de +10 en el futuro vale cada vez menos:

```
+10 en 1 paso     = 10 puntos
+10 en 2 pasos    = 10 × 0.9 = 9 puntos
+10 en 3 pasos    = 10 × 0.9 × 0.9 = 8.1 puntos
+10 en 10 pasos   = 10 × 0.9^9 ≈ 3.9 puntos
```

> 💡 **En Frozen Lake:** Como la única recompensa positiva (+1) está al final, necesitamos un gamma alto, entre 0.9 y 0.99 para que el agente sea lo suficientemente paciente como para planificar varios pasos hacia adelante. Si gamma fuera 0, el agente sería completamente indiferente a llegar a la meta porque "eso es muy lejano".

---

## 🧩 ¿Cómo Encajan Todas las Piezas?

Veamos el flujo completo de cómo el agente usa estos conceptos:

```
1. 📍 ESTADO: "Estoy en la casilla 6"

2. 📋 POLÍTICA: "Mi manual dice que desde la casilla 6 debo ir ABAJO"

3. 🎯 ACCIÓN: El agente ejecuta "ir abajo"

4. 🌍 ENTORNO: Responde con:
   - Nuevo estado: "Ahora estás en la casilla 10"
   - Recompensa: 0 (aún no llegas a la meta)

5. 🧠 APRENDIZAJE: "Hmm, ir abajo desde 6 me llevó a 10 con recompensa 0,
                    pero me acerca a la meta (recompensa futura +1)"

6. 🔄 REPETIR desde el paso 1 con el nuevo estado
```

**El objetivo final del agente:**

> Aprender la mejor política posible (π*) que maximice la suma de recompensas futuras, teniendo en cuenta que las recompensas lejanas valen un poco menos (según γ).

---

## ✅ Recapitulación

Hagamos un repaso visual de lo que hemos aprendido:

| **Concepto** | **Pregunta que responde** | **En Frozen Lake** |
|--------------|---------------------------|-------------------|
| **Estado** | ¿Dónde estoy? | Casilla del 0 al 15 |
| **Acción** | ¿Qué puedo hacer? | ←, ↓, →, ↑ (4 opciones) |
| **Recompensa** | ¿Qué tan bien lo hice? | +1 en meta, 0 resto |
| **Política** | ¿Qué debo hacer aquí? | El "manual" que queremos aprender |
| **Retorno** | ¿Cuánto voy a ganar total? | Suma de recompensas futuras |
| **Gamma (γ)** | ¿Cuánto valoro el futuro? | 0.9 típico (paciente pero realista) |

---

## 🚀 Siguiente Paso: La Q-Table

Ahora surge la pregunta del millón:

> **"¡Genial! Necesito una política óptima. Pero... ¿cómo diablos la encuentro si nadie me la va a dar?"**

La respuesta está en la **Q-Table**: una especie de "hoja de trucos" que el agente va escribiendo poco a poco mientras explora el lago congelado.

Esta tabla mágica guardará, para cada casilla y cada acción posible, una estimación de "qué tan buena es esa acción". Con miles de intentos, esos números se irán refinando hasta que el agente descubra cuál es la mejor estrategia.

En la siguiente sección descubriremos cómo funciona esta tabla y por qué es tan poderosa.

❄️ **¿Listo para conocer la memoria del agente?** ❄️

---

# **4. La Q-Table: La Memoria del Agente**

Imagina que estás explorando una ciudad nueva sin GPS. Decides llevar un cuaderno donde anotas tus descubrimientos:

```
📝 MI CUADERNO DE NAVEGACIÓN:

Plaza Central → Girar derecha: "Me llevó al museo ⭐⭐⭐⭐⭐"
Plaza Central → Seguir recto: "Callejón sin salida ⭐"
Parque → Ir al norte: "¡El mejor café! ⭐⭐⭐⭐⭐"
```

Después de varios días explorando, tu cuaderno se llena. Ya no preguntas direcciones, solo consultas tus notas y eliges el camino con más estrellas.

Eso es exactamente la **Q-Table**: el cuaderno del agente donde anota qué tan buena es cada decisión en cada lugar del lago.

---

## 🗺️ ¿Qué es la Q-Table?

Una tabla simple que responde: **"Si estoy aquí y hago esto, ¿qué tan bueno es?"**

- **Q** viene de **Quality** (calidad)
- En Frozen Lake: **16 filas** (estados) × **4 columnas** (acciones)
- Cada celda guarda un número que predice el éxito de esa decisión

```
           ← Izq   ↓ Abajo   → Der   ↑ Arr
Estado 0:    ?        ?        ?       ?
Estado 1:    ?        ?        ?       ?
  ...
Estado 15:   ?        ?        ?       ?
```

---

## 💡 Interpretando los Valores Q

Después de entrenar, supongamos que el estado 0 tiene estos valores:

```
Estado 0:  ← 0.71   ↓ 0.74   → 0.78   ↑ 0.68
```

**¿Qué significan estos números?**

Son predicciones de recompensa total futura. En Frozen Lake (con resbalones):

```
Q = 0.00  →  "Esto no funciona" 💀
Q = 0.30  →  "~30% de probabilidad de llegar a la meta"
Q = 0.74  →  "Buen camino, ~74% de éxito" 😊
Q = 0.95  →  "¡Casi óptimo!" 🎯
```

El agente simplemente elige la acción con el **valor Q más alto**: en este caso, **→ Derecha (0.78)**.

---

## 🛠️ Creando la Q-Table: El Cuaderno en Blanco

Al principio, el agente no sabe nada. Por eso iniciamos con **ceros**:

### 💻 Código: Inicialización

```python
import numpy as np
import gymnasium as gym

# Creamos el entorno
env = gym.make('FrozenLake-v1', map_name="4x4", is_slippery=True)

# Dimensiones
n_states = env.observation_space.n      # 16 casillas
n_actions = env.action_space.n          # 4 direcciones

# Inicializamos con ceros
Q_table = np.zeros((n_states, n_actions))

print(f"🧊 Q-Table creada: {Q_table.shape}")
print("\n🔍 Primeras filas:")
print(Q_table[:5])
```

**Salida:**
```
🧊 Q-Table creada: (16, 4)

🔍 Primeras filas:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
```

**¿Por qué ceros?** Porque al principio "todas las opciones parecen igualmente desconocidas". El agente explorará para aprender.

---

## 🎮 Usando la Q-Table para Decidir

Una vez entrenada, es muy simple:

```python
# Estamos en el estado 6
estado_actual = 6

# Consultamos la Q-Table
valores_q = Q_table[estado_actual]
print(f"📊 Valores Q: ← {valores_q[0]:.2f}  ↓ {valores_q[1]:.2f}  → {valores_q[2]:.2f}  ↑ {valores_q[3]:.2f}")

# Elegimos la mejor acción
mejor_accion = np.argmax(valores_q)
nombres = {0: "←", 1: "↓", 2: "→", 3: "↑"}
print(f"🎯 Decisión: {nombres[mejor_accion]}")
```

**Salida después del entrenamiento:**
```
📊 Valores Q: ← 0.25  ↓ 0.53  → 0.19  ↑ 0.31
🎯 Decisión: ↓
```

---

## 🧠 De la Q-Table a la Política

La conexión clave:

```
Q-Table (memoria)  →  Política (estrategia)
   ↓                       ↓
"Puntuaciones de      "Elige siempre
 cada acción"          el máximo"
```

Extrayendo la política es simple:

```python
# Para cada estado, toma la mejor acción
politica = np.argmax(Q_table, axis=1)

print("📋 Política aprendida:")
simbolos = {0: "←", 1: "↓", 2: "→", 3: "↑"}
for estado in [0, 1, 2, 6, 10, 14]:
    print(f"   Estado {estado:2d}: {simbolos[politica[estado]]}")
```

**Salida:**
```
📋 Política aprendida:
   Estado  0: →
   Estado  1: →
   Estado  2: ↓
   Estado  6: ↓
   Estado 10: ↓
   Estado 14: →
```

---

## 🌱 Evolución: Del Caos al Conocimiento

Lo fascinante es ver cómo cambia la Q-Table con el aprendizaje. Veamos el estado 14 (justo antes de la meta):

```
Episodio 0:       [0.00, 0.00, 0.00, 0.00]  ← "No sé nada"
Episodio 100:     [0.12, 0.08, 0.35, 0.18]  ← "Derecha parece mejor"
Episodio 1,000:   [0.21, 0.15, 0.78, 0.29]  ← "¡Derecha definitivamente!"
Episodio 10,000:  [0.24, 0.11, 0.93, 0.31]  ← "Casi óptimo"
```

> 💡 **La magia:** Con miles de intentos, los números reflejan la realidad. Desde el estado 14, ir a la derecha casi siempre lleva a la meta (+1).

---

## ✅ Recapitulación

| **Concepto** | **Explicación** | **En Frozen Lake** |
|--------------|-----------------|-------------------|
| **Q-Table** | Cuaderno de notas del agente | Tabla 16×4 |
| **Valor Q(s,a)** | "Calidad" de hacer *a* en *s* | Número entre 0 y 1 |
| **Inicialización** | Partir de cero (sin conocimiento) | `np.zeros((16, 4))` |
| **Decisión** | Elegir acción con Q más alto | `np.argmax(Q[estado])` |
| **Política** | Regla que dice qué hacer | Se deriva de la Q-Table |

---

## 🚀 Siguiente Paso

Ya tenemos la tabla... pero está llena de ceros. **¿Cómo aprende el agente a llenarla?**

La respuesta: **La Ecuación de Bellman**

Es una regla simple que dice: *"Actualiza lo que creías saber con lo que acabas de descubrir"*

Como cuando escribes en tu cuaderno de viaje: *"Pensaba que esta calle valía ⭐⭐⭐, pero acabo de encontrar el museo ⭐⭐⭐⭐⭐ → actualizo mi nota"*

En la siguiente sección descubriremos esta ecuación y los "mandos de control" del aprendizaje (α y γ).

❄️ **¿Listo para ver cómo los números cobran vida?** ❄️

---

# **5. La Ecuación de Bellman: ¿Cómo Aprende el Cerebro del Agente?**

Tenemos nuestra Q-Table lista, pero está llena de ceros. **¿Cómo se llenan esos números?**

La respuesta: **La Ecuación de Bellman**, la fórmula que permite al agente actualizar sus valores Q cada vez que experimenta algo.

---

## 🧠 La Idea Intuitiva

Imagina que anotas en tu cuaderno:

```
📝 "Plaza Central → Derecha = ⭐⭐⭐"
```

Pruebas esa ruta y te lleva a un museo increíble. Actualizas:

```
📝 "Plaza Central → Derecha = ⭐⭐⭐⭐ (mejoré mi estimación)"
```

Eso es la Ecuación de Bellman: **actualiza lo que creías con lo que acabas de descubrir**.

---

## 🎯 La Fórmula (sin asustarse)

$$Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \right]$$

En palabras simples:

```
Nuevo valor = Valor anterior + un poquito del error que cometí
```

---

## 🧩 Desglosando con un Ejemplo Real

Supongamos:
- Estado actual: **14** (casilla antes de la meta)
- Acción: **→ Derecha**
- Resultado: Estado **15** (¡META!), Recompensa **+1**

**Paso a paso:**

```
1. Valor actual en la Q-Table:
   Q(14, →) = 0.30  ← "Creía que valía 0.30"

2. Recompensa inmediata:
   r = 1  ← "¡Llegué a la meta!"

3. Mejor valor desde el nuevo estado:
   max Q(15, a') = 0  ← "La meta no tiene futuro"
   
4. Objetivo (lo que debería valer):
   Objetivo = r + γ × max Q(15)
            = 1 + 0.95 × 0
            = 1.0

5. Error de predicción:
   Error = Objetivo - Q_actual
         = 1.0 - 0.30
         = 0.70  ← "¡Subestimé esta acción!"

6. Actualización (con α = 0.1):
   Q_nuevo = 0.30 + 0.1 × 0.70
           = 0.30 + 0.07
           = 0.37
```

**Resultado:** El valor subió de 0.30 → 0.37. Pequeño ajuste, dirección correcta.

---

## ⚙️ Los Dos "Mandos de Control"

### 🎚️ α (Alpha) - Ritmo de Aprendizaje

Controla **qué tan rápido** aprende el agente:

| **α** | **Comportamiento** | **Analogía** |
|-------|-------------------|--------------|
| **0.01** | Aprende MUY lento | "Modifico la receta apenas un poquito" |
| **0.1** ✅ | **Equilibrado** | "Aprendo gradualmente" (recomendado) |
| **0.5** | Aprende rápido | "Me adapto rápido, pero puedo ser inestable" |
| **0.9** | MUY rápido | "Un error me hace cambiar todo" (arriesgado) |

> 💡 **Regla práctica:** Usa **α = 0.1** en Frozen Lake. Es estable y robusto al ruido.

### 🔮 γ (Gamma) - Visión de Futuro

Controla **cuánto valora** las recompensas futuras:

| **γ** | **Personalidad** | **En Frozen Lake** |
|-------|------------------|-------------------|
| **0.0** | Totalmente miope | ❌ Solo ve recompensa 0 inmediata |
| **0.9** | Equilibrado | ✅ Funciona bien, pero algo cortoplacista |
| **0.95-0.99** ✅ | **Muy recomendado** | ✅ Planea el camino completo (lo más usado) |

> 💡 **Regla práctica:** Empieza con **γ = 0.95** o **0.99** para Frozen Lake. Con valores bajos el agente es demasiado impulsivo.

**¿Por qué γ alto?** Como la recompensa +1 está al final del camino (4-6 pasos), necesitamos que el agente valore ese futuro lejano. Con γ = 0.9, las recompensas se descuentan mucho; con γ = 0.99, se conservan mejor.

---

## 💻 Código: La Ecuación en Una Línea

```python
# Parámetros
alpha = 0.1    # Ritmo de aprendizaje
gamma = 0.95   # Visión a futuro

# Ejemplo: estado 14 → derecha → meta (+1)
state = 14
action = 2         # → Derecha
new_state = 15     # Meta
reward = 1

# LA ECUACIÓN DE BELLMAN
Q_table[state, action] = Q_table[state, action] + alpha * (
    reward + gamma * np.max(Q_table[new_state]) - Q_table[state, action]
    #                ↑ La mejor acción desde el nuevo estado
)

print(f"✅ Valor actualizado: {Q_table[state, action]:.2f}")
```

**Salida:**
```
✅ Valor actualizado: 0.40
```

---

## 🌊 La Propagación del Conocimiento

Lo fascinante: el éxito en la meta se propaga **hacia atrás** como ondas:

```
Episodio 1:  Estado 14 → Meta (+1)
             Q(14, →) ≈ 0.10

Episodio 10: Estado 10 → Estado 14
             Q(10, ↓) aprende del valor de Q(14)
             
Episodio 50: Estado 6 → Estado 10
             Q(6, ↓) aprende del valor de Q(10)
```

**Como ondas en el agua:** El +1 de la meta crea ondas que iluminan el camino correcto hacia atrás, paso a paso.

---

## 🧪 Visualizando el Aprendizaje

Veamos cómo converge un valor con múltiples actualizaciones:

```python
# Simulación: 30 episodios exitosos desde estado 14 → meta
Q = 0.0
alpha = 0.1
gamma = 0.95

print("Episodio | Q(14, →)")
print("-" * 20)
for i in range(30):
    # Objetivo = 1 + 0.95 × 0 = 1.0
    Q = Q + alpha * (1.0 - Q)
    if i < 10 or i % 5 == 0:  # Mostramos los primeros 10 y luego cada 5
        print(f"   {i+1:2d}    | {Q:.3f}")
```

**Salida:**
```
Episodio | Q(14, →)
--------------------
    1    | 0.100
    2    | 0.190
    3    | 0.271
    4    | 0.344
    5    | 0.410
   10    | 0.651
   15    | 0.794
   20    | 0.878
   25    | 0.928
   30    | 0.958
```

> 💡 **Observa:** Converge gradualmente hacia 1.0. Con γ = 0.99 la convergencia es más lenta pero llega aún más cerca del valor óptimo teórico (1.0).

---

## 🛷 Robustez al Ruido: El Poder del α Bajo

En Frozen Lake el agente resbala ~30% del tiempo. **¿Cómo maneja Q-Learning esto?**

Ejemplo desde estado 14:

| **Intento** | **Resultado** | **Recompensa** | **Q(14, →)** |
|-------------|---------------|----------------|-------------|
| 1 | ✅ Meta | +1 | 0.00 → 0.10 |
| 2 | 🛷 Resbalón | 0 | 0.10 → 0.09 |
| 3 | ✅ Meta | +1 | 0.09 → 0.18 |
| 4 | 🛷 Resbalón | 0 | 0.18 → 0.16 |
| ... | ... | ... | ... |
| 100 | 70% éxito | Promedio | **≈ 0.70** |

**Resultado final:** Q converge a ~0.70, reflejando la probabilidad real de éxito (70%).

> 💡 **Esto es aprendizaje robusto:** El α bajo promedia las experiencias. Los resbalones no arruinan el aprendizaje, simplemente se incorporan al cálculo. Por eso α = 0.1 funciona tan bien en entornos con ruido.

---

## ✅ Recapitulación

| **Componente** | **Significado** | **Valor típico** |
|----------------|-----------------|------------------|
| **Q(s, a)** | Valor actual | Varía |
| **r** | Recompensa inmediata | 0 o 1 |
| **γ (gamma)** | Visión de futuro | 0.95 - 0.99 |
| **α (alpha)** | Ritmo de aprendizaje | 0.1 |
| **max Q(s', a')** | Mejor futuro posible | Del nuevo estado |
| **Error** | Objetivo - Q_actual | Se calcula |

**En palabras:**
> "Actualiza tu creencia sumando una fracción (α) de la diferencia entre lo que descubriste y lo que creías"

---

## 🚀 Siguiente Paso: ¿Explorar o Explotar?

Ya sabemos **cómo** actualizar valores. Pero surge un problema:

> Si al principio todos los Q valen 0... ¿cómo decide el agente qué acciones probar?

Si siempre elige el máximo (todas valen 0), ¡se queda atascado en la primera acción!

La solución: **ε-greedy** (epsilon-greedy), que equilibra:
- **Explotar:** Hacer lo que sé que funciona
- **Explorar:** Probar algo nuevo por si es mejor

En la siguiente sección descubriremos este dilema crucial y cómo resolverlo.

❄️ **¿Listo para el modo aventurero del agente?** ❄️

---